# 校則なんでも相談ボット

新入生の質問に、校則をもとに答えるボットです。下のセルを実行すると、入力欄が出ます。

答えを作るのは [LLM-jp](https://llm-jp.nii.ac.jp/) の日本語AIです。画面は [library-hiroba](https://pypi.org/project/library-hiroba/) で作っています。

しくみは2段階です。まず質問に近い条文を校則からさがし、次にその条文だけをAIに渡してやさしい言葉に直してもらいます。校則ぜんぶを渡さないのがポイントで、この作り方を RAG（検索してから答える方式）と呼びます。

> Google Colab で動かしてください。AIの計算に PyTorch を使うため、ブラウザ内で動く PyHiroba では実行できません。

In [ ]:
# ===== 校則なんでも相談ボット =====================================
# LLM-jp の日本語AIに、校則をもとに新入生の質問へ答えてもらいます。
# 画面は library-hiroba で作ります（サーバーは使いません）。
# ※ Google Colab で実行してください。AIの計算に PyTorch を使います。

%pip install -q library-hiroba

import torch, transformers
from library_hiroba import ui
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

MODEL_NAME = "llm-jp/llm-jp-3-980m-instruct3"
# 下の行を書きかえると、大きいAIと答えを比べられます
#   llm-jp/llm-jp-3-150m-instruct3 … 1.5億　約0.3GB
#   llm-jp/llm-jp-3-440m-instruct3 … 4.4億　約0.9GB
#   llm-jp/llm-jp-3-980m-instruct3 … 9.9億　約2.0GB

MIN_SCORE = 0.15   # これより近さが小さいと「条文が見つからなかった」と伝えます

# ----- 1. 校則（教材用に作成した架空の規程です） -----------------------
SCHOOL_RULES = [
    ("第3条（登校時刻）", "朝は8時30分までに登校します。おくれた場合は遅刻になります。電車がおくれたときは、遅延証明書を出せば遅刻になりません。"),
    ("第4条（欠席の届出）", "休むとき、おくれるとき、早く帰るときは、8時15分までに保護者から学校に連絡します。3日以上続けて休むときは、医師の診断書が必要です。"),
    ("第5条（服装）", "登下校と校内では制服を着ます。体育の授業では体操服を着ます。制服を短くするなど、作りかえることはできません。"),
    ("第6条（衣替えと防寒具）", "夏服への切りかえは6月1日から15日まで、冬服への切りかえは10月1日から15日までです。冬はコートやマフラーを使えます。ただし校舎の中ではコートをぬぎます。"),
    ("第7条（頭髪と装飾品）", "髪は清潔にします。髪を染めることとパーマはできません。生まれつきの髪の色は、入学のときに担任に伝えます。化粧とピアスはできません。"),
    ("第8条（持ち物）", "勉強に必要のない物は持ってきません。多くのお金や高い物は持ってきません。なくしたり盗まれたりしたときは、すぐに担任に伝えます。"),
    ("第9条（電子機器）", "スマートフォンや携帯電話を学校に持ってくることはできます。ただし授業のある時間は、電源を切って鞄にしまいます。先生が使ってよいと言ったときは使えます。人を撮るときは、その人の許可をとります。"),
    ("第10条（通学の方法）", "歩くか電車やバスで通学します。自動車やバイクでの通学はできません。バイクや車の免許を取ることも、在学中はできません。"),
    ("第11条（自転車通学）", "自転車で通学するには、申し込んで学校の許可をもらいます。許可証を自転車にはります。ヘルメットをかぶり、保険に入ります。自転車は駐輪場にとめて鍵をかけます。"),
    ("第12条（アルバイト）", "アルバイトは原則できません。家庭の事情があるときは、保護者の同意を得て申し込み、学校の許可をもらえばできます。夜10時から朝5時までの仕事はできません。"),
    ("第13条（外泊と旅行）", "生徒だけで外泊したり泊まりがけで旅行したりするときは、保護者の同意を得て、前もって担任に届け出ます。長い休みの間も同じです。"),
    ("第14条（インターネット）", "インターネットで発信するとき、人を傷つけたり学校の名誉を傷つけたりしてはいけません。校内で撮った写真や動画を、写っている人の許可なくネットに出してはいけません。"),
    ("第15条（部活動）", "部活動に入るか入らないかは自由です。平日は午後6時までです。日曜日と祝日は原則として活動しません。定期テストの1週間前から、テストが終わるまで活動はありません。"),
    ("第16条（きまりを守れないとき）", "きまりを守れないときは、担任や先生が指導します。それでも直らないときは、保護者を交えて話し合います。"),
]

ALIASES = {
    "第3条": "遅刻 ちこく 何時 朝 電車 バス 間に合わない",
    "第4条": "休む 休んだら 欠席 やすむ 風邪 体調不良 早退 連絡 保護者 何日 続けて 診断書",
    "第5条": "制服 服装 私服 ボタン 着る 体育",
    "第6条": "コート マフラー 手袋 上着 寒い 暑い 衣替え 夏服 冬服",
    "第7条": "髪 かみ 髪型 染める 色 パーマ ピアス 化粧 メイク 地毛",
    "第8条": "持ち物 財布 お金 現金 貴重品 なくした 盗まれた ゲーム",
    "第9条": "スマホ スマートフォン 携帯 ケータイ 電話 充電 イヤホン 音楽 撮影 写真 動画 授業中",
    "第10条": "通学 徒歩 電車 バス バイク 原付 車 免許",
    "第11条": "自転車 チャリ 駐輪場 ヘルメット 保険 許可証 鍵",
    "第12条": "アルバイト バイト 働く 仕事 お金を稼ぐ 深夜",
    "第13条": "外泊 旅行 泊まり 友達の家 長期休み 夏休み",
    "第14条": "SNS インスタ X ツイッター 投稿 ネット 写真 動画 悪口",
    "第15条": "部活 部活動 練習 日曜 祝日 テスト前 下校時刻 何時まで 活動",
    "第16条": "違反 破ったら 守れない 指導 呼び出し 反省",
}

# ----- 2. 質問に近い条文をさがす（ここはAIを使いません） ---------------
search_texts = [f"{t} {b} {ALIASES.get(t.split('（')[0], '')}" for t, b in SCHOOL_RULES]
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 2), sublinear_tf=True)
rule_vectors = vectorizer.fit_transform(search_texts)

def find_rule(question):
  scores = cosine_similarity(vectorizer.transform([question]), rule_vectors)[0]
  i = int(scores.argmax())
  return (*SCHOOL_RULES[i], float(scores[i]))

# ----- 3. AIを読み込む（少し時間がかかります） -------------------------
print(f"AIを読み込んでいます… {MODEL_NAME}")
use_gpu = torch.cuda.is_available()
dtype = torch.float16 if use_gpu else torch.float32
dtype_key = "dtype" if int(transformers.__version__.split(".")[0]) >= 5 else "torch_dtype"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", **{dtype_key: dtype})
model.eval()

# ----- 4. AIに渡す文をつくる（お手本を1組見せてから質問します） --------
SHOT_RULE = "スマートフォンや携帯電話を学校に持ってくることはできます。ただし授業のある時間は、電源を切って鞄にしまいます。"
SHOT_Q = "スマホは持っていっていいですか。"
SHOT_A = "はい、持ってきていいです。ただし授業の時間は電源を切って鞄にしまってください。"
HEAD = "校則をもとに、新入生の質問にやさしい言葉で答えてください。\n"

def build_inputs(body, question):
  messages = [
    {"role": "user", "content": f"{HEAD}校則：{SHOT_RULE}\n質問：{SHOT_Q}"},
    {"role": "assistant", "content": SHOT_A},
    {"role": "user", "content": f"校則：{body}\n質問：{question}"},
  ]
  encoded = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
  inputs = {"input_ids": encoded} if hasattr(encoded, "shape") else dict(encoded)
  return {k: v.to(model.device) for k, v in inputs.items()}

def trim(reply):
  for stop in ["校則：", "質問：", "答え：", "###"]:
    if stop in reply:
      reply = reply.split(stop)[0]
  return reply.strip()

# ----- 5. 質問に答えて、会話に足す -------------------------------------
history = []   # ここに会話がたまっていきます

def ask(question):
  if not question.strip():
    return ui.chat(history) if history else ui.alert("聞きたいことを入力してください。")

  title, body, score = find_rule(question)
  inputs = build_inputs(body, question)
  length = inputs["input_ids"].shape[-1]

  with torch.no_grad():
    generated = model.generate(**inputs, max_new_tokens=100, do_sample=False,
      repetition_penalty=1.15, no_repeat_ngram_size=3, pad_token_id=tokenizer.eos_token_id)
  reply = trim(tokenizer.decode(generated[0][length:], skip_special_tokens=True))
  if not reply:
    reply = "うまく答えられませんでした。質問の書き方を変えてみてください。"

  # 答えの下に「読んだ条文」と「渡した文の長さ」を添えます
  detail = ui.columns(
    ui.reveal(body, summary=f"読んだ条文：{title}"),
    ui.stat("渡した文の長さ", length, unit="トークン"),
    widths=[3, 1])

  history.append({"role": "user", "content": question})
  history.append({"role": "assistant", "content": ui.stack(reply, detail)})
  if score < MIN_SCORE:
    history.append({"role": "note",
      "content": "この質問に近い条文は校則にありませんでした。答えはAIの推測かもしれません。"})
  return ui.chat(history, names={"user": "あなた", "assistant": "校則ボット"})

# ----- 6. 画面を出す ---------------------------------------------------
ui.form(ask,
        ui.field("question", label="質問", placeholder="スマホって学校に持っていっていいの？"),
        title="校則なんでも相談ボット", submit_label="聞く", clear_on_submit=True)

## やってみよう

1. `MODEL_NAME` を小さいAI（`llm-jp-3-150m-instruct3`）に変えて、答えを比べてみましょう
2. `SCHOOL_RULES` に自分の学校の校則を1つ足して、その条文について質問してみましょう
3. `ALIASES` に言葉を足すと、条文が見つかりやすくなります
4. `MIN_SCORE` を大きくすると、注意が出やすくなります

## 考えてみよう

- AIは校則に書いていないことも、それらしく答えてしまうことがあります。どうすれば気づけるでしょうか
- 「読んだ条文」を画面に出しているのはなぜでしょうか
- このボットの答えだけを信じて行動してよいでしょうか